# Playground Series S6E3: Telco Seed-Ensemble XGBoost

**Competition:** [Playground Series S6E3](https://www.kaggle.com/competitions/playground-series-s6e3)  
**Goal:** run the exact multi-seed telco XGBoost path from the repo on Kaggle so the heavy benchmark does not depend on local CPU.  
**Author:** Lorenzo Scaturchio

---

## Plan

1. Load the competition train/test and the original IBM telco churn dataset.
2. Build the same advanced telco feature frame used in the local competition lab.
3. Train the seed-averaged nested target-encoded XGBoost ensemble.
4. Report the full OOF AUC and write `submission.csv`.

This notebook is intentionally narrow: it is for leaderboard movement, not tutorial exposition.

## 1. Setup

In [ ]:
import json
import warnings
from itertools import combinations
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
RANDOM_STATE = 42
print('Environment ready.')

## 2. Data Loading

In [ ]:
def find_input_file(filename: str) -> Path | None:
    candidates = [
        Path('/kaggle/input/playground-series-s6e3') / filename,
        Path('/kaggle/input/competitions/playground-series-s6e3') / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob(filename))
        if matches:
            return matches[0]
    return None

def find_original_telco_file() -> Path | None:
    candidates = [
        Path('/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
        Path('/kaggle/input/wa-fnusec-telcocustomerchurn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
        Path('/kaggle/input/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob('WA_Fn-UseC_-Telco-Customer-Churn.csv'))
        if matches:
            return matches[0]
    return None

train_path = find_input_file('train.csv')
test_path = find_input_file('test.csv')
orig_path = find_original_telco_file()

if train_path is None or test_path is None or orig_path is None:
    raise FileNotFoundError('Required competition or original telco files were not found under /kaggle/input.')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
orig = pd.read_csv(orig_path)

print(f'train: {train.shape}')
print(f'test : {test.shape}')
print(f'orig : {orig.shape}')
print(f'competition train path: {train_path}')
print(f'original telco path  : {orig_path}')

## 3. Embedded Competition-Lab Functions

In [ ]:
def _concat_feature_block(df: pd.DataFrame, updates: dict[str, Any]) -> pd.DataFrame:
    if not updates:
        return df
    block = pd.DataFrame(updates, index=df.index)
    return pd.concat([df, block], axis=1).copy()


def _playground_advanced_feature_frames(
    train: pd.DataFrame,
    test: pd.DataFrame,
    orig: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], list[str]]:
    def pctrank_against(values: np.ndarray, reference: np.ndarray) -> np.ndarray:
        ref = np.sort(np.asarray(reference, dtype=np.float32))
        if ref.size == 0:
            return np.zeros(len(values), dtype=np.float32)
        return (np.searchsorted(ref, values, side="left") / ref.size).astype(np.float32)

    def zscore_against(values: np.ndarray, reference: np.ndarray) -> np.ndarray:
        ref = np.asarray(reference, dtype=np.float32)
        if ref.size == 0:
            return np.zeros(len(values), dtype=np.float32)
        sigma = float(ref.std())
        if sigma == 0.0 or np.isnan(sigma):
            return np.zeros(len(values), dtype=np.float32)
        return ((values - float(ref.mean())) / sigma).astype(np.float32)

    train = train.copy()
    test = test.copy()
    orig = orig.copy()
    if "customerID" in orig.columns:
        orig = orig.drop(columns=["customerID"])

    target = "Churn"
    train[target] = (
        train[target].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0}).fillna(train[target]).astype(int)
    )
    orig[target] = (
        orig[target].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0}).fillna(orig[target]).astype(int)
    )
    cat_cols = [
        "gender",
        "SeniorCitizen",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
    ]
    num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
    service_cols = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]

    for df in (train, test, orig):
        for col in cat_cols:
            df[col] = df[col].astype(str).fillna("missing").str.strip()
        for col in num_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
            df[col] = df[col].fillna(df[col].median())

    new_num_cols: list[str] = []
    freq_maps = {
        col: pd.concat([train[col], test[col], orig[col]], axis=0).value_counts(normalize=True)
        for col in num_cols
    }
    train = _concat_feature_block(
        train,
        {f"FREQ_{col}": train[col].map(freq_maps[col]).fillna(0).astype("float32") for col in num_cols},
    )
    test = _concat_feature_block(
        test,
        {f"FREQ_{col}": test[col].map(freq_maps[col]).fillna(0).astype("float32") for col in num_cols},
    )
    orig = _concat_feature_block(
        orig,
        {f"FREQ_{col}": orig[col].map(freq_maps[col]).fillna(0).astype("float32") for col in num_cols},
    )
    new_num_cols.extend([f"FREQ_{col}" for col in num_cols])

    all_num = pd.concat([train[num_cols], test[num_cols], orig[num_cols]], axis=0, ignore_index=True)
    rank_updates_train: dict[str, Any] = {}
    rank_updates_test: dict[str, Any] = {}
    rank_updates_orig: dict[str, Any] = {}
    for col in num_cols:
        ranks = all_num[col].rank(method="average", pct=True).astype("float32").to_numpy()
        rank_updates_train[f"RANK_{col}"] = ranks[: len(train)]
        rank_updates_test[f"RANK_{col}"] = ranks[len(train) : len(train) + len(test)]
        rank_updates_orig[f"RANK_{col}"] = ranks[len(train) + len(test) :]
    train = _concat_feature_block(train, rank_updates_train)
    test = _concat_feature_block(test, rank_updates_test)
    orig = _concat_feature_block(orig, rank_updates_orig)
    new_num_cols.extend([f"RANK_{col}" for col in num_cols])

    def _power_updates(df: pd.DataFrame) -> dict[str, Any]:
        updates: dict[str, Any] = {}
        for col in num_cols:
            values = df[col].astype("float32")
            updates[f"LOG1P_{col}"] = np.log1p(values.clip(lower=0)).astype("float32")
            updates[f"SQRT_{col}"] = np.sqrt(values.clip(lower=0)).astype("float32")
            updates[f"INV1P_{col}"] = (1.0 / (1.0 + values.clip(lower=0))).astype("float32")
        return updates

    train = _concat_feature_block(train, _power_updates(train))
    test = _concat_feature_block(test, _power_updates(test))
    orig = _concat_feature_block(orig, _power_updates(orig))
    new_num_cols.extend([f"LOG1P_{col}" for col in num_cols])
    new_num_cols.extend([f"SQRT_{col}" for col in num_cols])
    new_num_cols.extend([f"INV1P_{col}" for col in num_cols])

    def _core_numeric_updates(df: pd.DataFrame) -> dict[str, Any]:
        charges_deviation = (df["TotalCharges"] - df["tenure"] * df["MonthlyCharges"]).astype("float32")
        service_yes_count = (df[service_cols] == "Yes").sum(axis=1).astype("float32")
        return {
            "charges_deviation": charges_deviation,
            "abs_charges_dev": np.abs(charges_deviation).astype("float32"),
            "monthly_to_total_ratio": (df["MonthlyCharges"] / (df["TotalCharges"] + 1)).astype("float32"),
            "total_to_monthly_ratio": (df["TotalCharges"] / (df["MonthlyCharges"] + 1)).astype("float32"),
            "avg_monthly_charges": (df["TotalCharges"] / (df["tenure"] + 1)).astype("float32"),
            "tenure_x_monthly": (df["tenure"] * df["MonthlyCharges"]).astype("float32"),
            "tenure_x_total": (df["tenure"] * df["TotalCharges"]).astype("float32"),
            "service_yes_count": service_yes_count,
            "service_no_count": (df[service_cols] == "No").sum(axis=1).astype("float32"),
            "service_other_count": (
                df[service_cols].isin(["No phone service", "No internet service"]).sum(axis=1).astype("float32")
            ),
            "service_count": service_yes_count,
            "has_internet": (df["InternetService"] != "No").astype("float32"),
            "has_phone": (df["PhoneService"] == "Yes").astype("float32"),
        }

    train = _concat_feature_block(train, _core_numeric_updates(train))
    test = _concat_feature_block(test, _core_numeric_updates(test))
    orig = _concat_feature_block(orig, _core_numeric_updates(orig))
    new_num_cols.extend(
        [
            "charges_deviation",
            "abs_charges_dev",
            "monthly_to_total_ratio",
            "total_to_monthly_ratio",
            "avg_monthly_charges",
            "tenure_x_monthly",
            "tenure_x_total",
            "service_yes_count",
            "service_no_count",
            "service_other_count",
            "service_count",
            "has_internet",
            "has_phone",
        ]
    )

    new_cat_cols: list[str] = []
    tenure_bins = [0, 1, 3, 6, 12, 24, 36, 48, 60, 72, 10_000]
    monthly_bins = pd.qcut(
        pd.concat([train["MonthlyCharges"], test["MonthlyCharges"], orig["MonthlyCharges"]]),
        q=40,
        retbins=True,
        duplicates="drop",
    )[1]
    total_bins = pd.qcut(
        pd.concat([train["TotalCharges"], test["TotalCharges"], orig["TotalCharges"]]),
        q=60,
        retbins=True,
        duplicates="drop",
    )[1]
    train = _concat_feature_block(
        train,
        {
            "tenure_bin": pd.cut(train["tenure"], bins=tenure_bins, include_lowest=True).astype(str),
            "MonthlyCharges_bin": pd.cut(train["MonthlyCharges"], bins=monthly_bins, include_lowest=True).astype(str),
            "TotalCharges_bin": pd.cut(train["TotalCharges"], bins=total_bins, include_lowest=True).astype(str),
        },
    )
    test = _concat_feature_block(
        test,
        {
            "tenure_bin": pd.cut(test["tenure"], bins=tenure_bins, include_lowest=True).astype(str),
            "MonthlyCharges_bin": pd.cut(test["MonthlyCharges"], bins=monthly_bins, include_lowest=True).astype(str),
            "TotalCharges_bin": pd.cut(test["TotalCharges"], bins=total_bins, include_lowest=True).astype(str),
        },
    )
    orig = _concat_feature_block(
        orig,
        {
            "tenure_bin": pd.cut(orig["tenure"], bins=tenure_bins, include_lowest=True).astype(str),
            "MonthlyCharges_bin": pd.cut(orig["MonthlyCharges"], bins=monthly_bins, include_lowest=True).astype(str),
            "TotalCharges_bin": pd.cut(orig["TotalCharges"], bins=total_bins, include_lowest=True).astype(str),
        },
    )
    new_cat_cols.extend(["tenure_bin", "MonthlyCharges_bin", "TotalCharges_bin"])

    yn_cols = [
        "Partner",
        "Dependents",
        "PhoneService",
        "PaperlessBilling",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "MultipleLines",
    ]
    def _yn_updates(df: pd.DataFrame) -> dict[str, Any]:
        updates: dict[str, Any] = {}
        for col in yn_cols:
            values = df[col].astype(str)
            updates[f"ISYES_{col}"] = (values == "Yes").astype("float32")
            updates[f"ISNO_{col}"] = (values == "No").astype("float32")
            updates[f"ISOTHER_{col}"] = (~values.isin(["Yes", "No"])).astype("float32")
        return updates

    train = _concat_feature_block(train, _yn_updates(train))
    test = _concat_feature_block(test, _yn_updates(test))
    orig = _concat_feature_block(orig, _yn_updates(orig))
    new_num_cols.extend([f"ISYES_{col}" for col in yn_cols])
    new_num_cols.extend([f"ISNO_{col}" for col in yn_cols])
    new_num_cols.extend([f"ISOTHER_{col}" for col in yn_cols])

    cat_feature_updates = {"train": {}, "test": {}, "orig": {}}
    for left, right in (
        ("Contract", "InternetService"),
        ("PaymentMethod", "Contract"),
        ("InternetService", "OnlineSecurity"),
        ("PaymentMethod", "PaperlessBilling"),
        ("Contract", "PaperlessBilling"),
        ("InternetService", "TechSupport"),
    ):
        name = f"{left}__{right}"
        cat_feature_updates["train"][name] = train[left].astype(str) + "|" + train[right].astype(str)
        cat_feature_updates["test"][name] = test[left].astype(str) + "|" + test[right].astype(str)
        cat_feature_updates["orig"][name] = orig[left].astype(str) + "|" + orig[right].astype(str)
        new_cat_cols.append(name)

    for left, middle, right in (("Contract", "InternetService", "PaymentMethod"),):
        name = f"{left}__{middle}__{right}"
        cat_feature_updates["train"][name] = (
            train[left].astype(str) + "|" + train[middle].astype(str) + "|" + train[right].astype(str)
        )
        cat_feature_updates["test"][name] = (
            test[left].astype(str) + "|" + test[middle].astype(str) + "|" + test[right].astype(str)
        )
        cat_feature_updates["orig"][name] = (
            orig[left].astype(str) + "|" + orig[middle].astype(str) + "|" + orig[right].astype(str)
        )
        new_cat_cols.append(name)

    ngram_top_cols = [
        "Contract",
        "InternetService",
        "PaymentMethod",
        "OnlineSecurity",
        "TechSupport",
        "PaperlessBilling",
    ]
    for left, right in combinations(ngram_top_cols, 2):
        name = f"BG_{left}_{right}"
        cat_feature_updates["train"][name] = train[left].astype(str) + "_" + train[right].astype(str)
        cat_feature_updates["test"][name] = test[left].astype(str) + "_" + test[right].astype(str)
        cat_feature_updates["orig"][name] = orig[left].astype(str) + "_" + orig[right].astype(str)
        new_cat_cols.append(name)

    for left, middle, right in combinations(ngram_top_cols[:4], 3):
        name = f"TG_{left}_{middle}_{right}"
        cat_feature_updates["train"][name] = (
            train[left].astype(str) + "_" + train[middle].astype(str) + "_" + train[right].astype(str)
        )
        cat_feature_updates["test"][name] = (
            test[left].astype(str) + "_" + test[middle].astype(str) + "_" + test[right].astype(str)
        )
        cat_feature_updates["orig"][name] = (
            orig[left].astype(str) + "_" + orig[middle].astype(str) + "_" + orig[right].astype(str)
        )
        new_cat_cols.append(name)
    train = _concat_feature_block(train, cat_feature_updates["train"])
    test = _concat_feature_block(test, cat_feature_updates["test"])
    orig = _concat_feature_block(orig, cat_feature_updates["orig"])

    counted_cat_cols = cat_cols + new_cat_cols
    all_cat_frame = pd.concat([train[counted_cat_cols], test[counted_cat_cols], orig[counted_cat_cols]], ignore_index=True)
    count_updates_train: dict[str, Any] = {}
    count_updates_test: dict[str, Any] = {}
    count_updates_orig: dict[str, Any] = {}
    for col in counted_cat_cols:
        counts = all_cat_frame[col].value_counts(dropna=False)
        train_counts = train[col].map(counts).fillna(0).astype("float32")
        test_counts = test[col].map(counts).fillna(0).astype("float32")
        orig_counts = orig[col].map(counts).fillna(0).astype("float32")
        count_updates_train[f"CAT_CNT_{col}"] = train_counts
        count_updates_test[f"CAT_CNT_{col}"] = test_counts
        count_updates_orig[f"CAT_CNT_{col}"] = orig_counts
        count_updates_train[f"CAT_RARE_{col}"] = (train_counts <= 50).astype("float32")
        count_updates_test[f"CAT_RARE_{col}"] = (test_counts <= 50).astype("float32")
        count_updates_orig[f"CAT_RARE_{col}"] = (orig_counts <= 50).astype("float32")
        new_num_cols.extend([f"CAT_CNT_{col}", f"CAT_RARE_{col}"])
    train = _concat_feature_block(train, count_updates_train)
    test = _concat_feature_block(test, count_updates_test)
    orig = _concat_feature_block(orig, count_updates_orig)

    orig_global = float(orig[target].mean())
    orig_proba_updates_train: dict[str, Any] = {}
    orig_proba_updates_test: dict[str, Any] = {}
    orig_proba_updates_orig: dict[str, Any] = {}
    for col in cat_cols + num_cols + new_cat_cols:
        lookup = orig.groupby(col, observed=False)[target].mean()
        name = f"ORIG_proba_{col}"
        orig_proba_updates_train[name] = train[col].map(lookup).fillna(orig_global).astype("float32")
        orig_proba_updates_test[name] = test[col].map(lookup).fillna(orig_global).astype("float32")
        orig_proba_updates_orig[name] = orig[col].map(lookup).fillna(orig_global).astype("float32")
        new_num_cols.append(name)
    train = _concat_feature_block(train, orig_proba_updates_train)
    test = _concat_feature_block(test, orig_proba_updates_test)
    orig = _concat_feature_block(orig, orig_proba_updates_orig)

    orig_churner_tc = orig.loc[orig[target] == 1, "TotalCharges"].to_numpy(dtype=np.float32)
    orig_nonchurner_tc = orig.loc[orig[target] == 0, "TotalCharges"].to_numpy(dtype=np.float32)
    orig_tc = orig["TotalCharges"].to_numpy(dtype=np.float32)
    orig_is_mc_mean = orig.groupby("InternetService", observed=False)["MonthlyCharges"].mean()
    distribution_cols = [
        "pctrank_nonchurner_TC",
        "pctrank_churner_TC",
        "pctrank_orig_TC",
        "zscore_churn_gap_TC",
        "zscore_nonchurner_TC",
        "pctrank_churn_gap_TC",
        "resid_IS_MC",
        "cond_pctrank_IS_TC",
        "cond_pctrank_C_TC",
    ]
    def _distribution_updates(df: pd.DataFrame) -> dict[str, Any]:
        tc = df["TotalCharges"].to_numpy(dtype=np.float32)
        updates: dict[str, Any] = {
            "pctrank_nonchurner_TC": pctrank_against(tc, orig_nonchurner_tc),
            "pctrank_churner_TC": pctrank_against(tc, orig_churner_tc),
            "pctrank_orig_TC": pctrank_against(tc, orig_tc),
            "zscore_churn_gap_TC": (
                np.abs(zscore_against(tc, orig_churner_tc)) - np.abs(zscore_against(tc, orig_nonchurner_tc))
            ).astype(np.float32),
            "zscore_nonchurner_TC": zscore_against(tc, orig_nonchurner_tc),
            "pctrank_churn_gap_TC": (
                pctrank_against(tc, orig_churner_tc) - pctrank_against(tc, orig_nonchurner_tc)
            ).astype(np.float32),
            "resid_IS_MC": (
                df["MonthlyCharges"] - df["InternetService"].map(orig_is_mc_mean).fillna(0).to_numpy(dtype=np.float32)
            ).astype(np.float32),
        }
        cond_is_vals = np.zeros(len(df), dtype=np.float32)
        for cat_val in orig["InternetService"].dropna().astype(str).unique():
            mask = df["InternetService"].astype(str) == cat_val
            if not mask.any():
                continue
            ref = orig.loc[orig["InternetService"].astype(str) == cat_val, "TotalCharges"].to_numpy(dtype=np.float32)
            cond_is_vals[mask.to_numpy()] = pctrank_against(
                df.loc[mask, "TotalCharges"].to_numpy(dtype=np.float32),
                ref,
            )
        updates["cond_pctrank_IS_TC"] = cond_is_vals

        cond_contract_vals = np.zeros(len(df), dtype=np.float32)
        for cat_val in orig["Contract"].dropna().astype(str).unique():
            mask = df["Contract"].astype(str) == cat_val
            if not mask.any():
                continue
            ref = orig.loc[orig["Contract"].astype(str) == cat_val, "TotalCharges"].to_numpy(dtype=np.float32)
            cond_contract_vals[mask.to_numpy()] = pctrank_against(
                df.loc[mask, "TotalCharges"].to_numpy(dtype=np.float32),
                ref,
            )
        updates["cond_pctrank_C_TC"] = cond_contract_vals
        return updates

    train = _concat_feature_block(train, _distribution_updates(train))
    test = _concat_feature_block(test, _distribution_updates(test))
    new_num_cols.extend(distribution_cols)

    num_as_cat: list[str] = []
    num_as_cat_updates_train: dict[str, Any] = {}
    num_as_cat_updates_test: dict[str, Any] = {}
    num_as_cat_updates_orig: dict[str, Any] = {}
    for col in num_cols:
        cat_name = f"CAT_{col}"
        num_as_cat.append(cat_name)
        num_as_cat_updates_train[cat_name] = train[col].astype(str)
        num_as_cat_updates_test[cat_name] = test[col].astype(str)
        num_as_cat_updates_orig[cat_name] = orig[col].astype(str)
    train = _concat_feature_block(train, num_as_cat_updates_train)
    test = _concat_feature_block(test, num_as_cat_updates_test)
    orig = _concat_feature_block(orig, num_as_cat_updates_orig)

    for df in (train, test, orig):
        for col in cat_cols + new_cat_cols + num_as_cat:
            df[col] = df[col].astype("category")

    feature_cols = num_cols + cat_cols + new_num_cols + new_cat_cols + num_as_cat
    te_cols = num_as_cat + cat_cols + new_cat_cols
    drop_raw_cols = num_as_cat + cat_cols + new_cat_cols
    return train, test, feature_cols, te_cols, drop_raw_cols


def _playground_advanced_xgboost_result(
    train: pd.DataFrame,
    test: pd.DataFrame,
    orig: pd.DataFrame,
    folds: int,
    seeds: tuple[int, ...] = (11, 42, 99),
) -> tuple[float, np.ndarray, np.ndarray]:
    target = "Churn"
    train_frame, test_frame, feature_cols, te_cols, drop_raw_cols = _playground_advanced_feature_frames(
        train,
        test,
        orig,
    )
    n_splits = min(max(3, folds), 5)
    inner_splits = min(3, n_splits)
    stats = ["std", "min", "max"]
    oof_sum = np.zeros(len(train_frame), dtype=float)
    oof_count = np.zeros(len(train_frame), dtype=float)
    test_pred = np.zeros(len(test_frame), dtype=float)
    total_models = 0

    try:
        import xgboost as xgb
    except ImportError as exc:
        raise RuntimeError("xgboost is not installed") from exc

    params = {
        "n_estimators": 6000,
        "learning_rate": 0.02,
        "max_depth": 5,
        "subsample": 0.81,
        "colsample_bytree": 0.55,
        "min_child_weight": 6,
        "reg_alpha": 1.25,
        "reg_lambda": 1.3,
        "gamma": 0.35,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "enable_categorical": True,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "early_stopping_rounds": 200,
    }

    for seed in seeds:
        outer_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for train_idx, valid_idx in outer_cv.split(train_frame, train_frame[target]):
            x_train = train_frame.iloc[train_idx][feature_cols + [target]].reset_index(drop=True).copy()
            y_train = train_frame.iloc[train_idx][target].to_numpy()
            y_valid = train_frame.iloc[valid_idx][target].to_numpy()
            x_valid = train_frame.iloc[valid_idx][feature_cols].reset_index(drop=True).copy()
            x_test = test_frame[feature_cols].reset_index(drop=True).copy()
            inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)

            te_stat_cols = [f"TE1_{col}_{stat}" for col in te_cols for stat in stats]
            x_train = _concat_feature_block(x_train, {name: np.nan for name in te_stat_cols})

            for inner_train_idx, inner_valid_idx in inner_cv.split(x_train, y_train):
                x_inner_train = x_train.loc[inner_train_idx, feature_cols + [target]].copy()
                x_inner_valid = x_train.loc[inner_valid_idx, feature_cols].copy()
                for col in te_cols:
                    grouped = x_inner_train.groupby(col, observed=False)[target].agg(stats)
                    grouped.columns = [f"TE1_{col}_{stat}" for stat in stats]
                    x_inner_valid = x_inner_valid.merge(grouped, on=col, how="left")
                    for name in grouped.columns:
                        x_train.loc[inner_valid_idx, name] = x_inner_valid[name].to_numpy(dtype="float32")

            for col in te_cols:
                grouped = x_train.groupby(col, observed=False)[target].agg(stats)
                grouped.columns = [f"TE1_{col}_{stat}" for stat in stats]
                x_valid = x_valid.merge(grouped.astype("float32"), on=col, how="left")
                x_test = x_test.merge(grouped.astype("float32"), on=col, how="left")
                for name in grouped.columns:
                    x_train[name] = x_train[name].fillna(0).astype("float32")
                    x_valid[name] = x_valid[name].fillna(0).astype("float32")
                    x_test[name] = x_test[name].fillna(0).astype("float32")

            if te_cols:
                mean_encoder = TargetEncoder(
                    cv=inner_splits,
                    shuffle=True,
                    smooth="auto",
                    target_type="binary",
                    random_state=seed,
                )
                mean_cols = [f"TE_{col}" for col in te_cols]
                x_train = pd.concat(
                    [
                        x_train,
                        pd.DataFrame(
                            mean_encoder.fit_transform(x_train[te_cols], y_train),
                            columns=mean_cols,
                            index=x_train.index,
                        ),
                    ],
                    axis=1,
                ).copy()
                x_valid = pd.concat(
                    [
                        x_valid,
                        pd.DataFrame(
                            mean_encoder.transform(x_valid[te_cols]),
                            columns=mean_cols,
                            index=x_valid.index,
                        ),
                    ],
                    axis=1,
                ).copy()
                x_test = pd.concat(
                    [
                        x_test,
                        pd.DataFrame(
                            mean_encoder.transform(x_test[te_cols]),
                            columns=mean_cols,
                            index=x_test.index,
                        ),
                    ],
                    axis=1,
                ).copy()

            for df in (x_train, x_valid, x_test):
                for col in te_cols:
                    df[col] = df[col].astype(str).astype("category")
                df.drop(columns=drop_raw_cols, inplace=True)
            x_train.drop(columns=[target], inplace=True)

            model = xgb.XGBClassifier(**params, random_state=seed)
            model.fit(
                x_train,
                y_train,
                eval_set=[(x_valid, y_valid)],
                verbose=False,
            )
            valid_pred = model.predict_proba(x_valid)[:, 1]
            oof_sum[valid_idx] += valid_pred
            oof_count[valid_idx] += 1.0
            test_pred += model.predict_proba(x_test)[:, 1]
            total_models += 1

    if total_models == 0 or np.any(oof_count == 0):
        raise RuntimeError("XGBoost did not produce a complete OOF prediction.")

    oof = oof_sum / oof_count
    test_pred = test_pred / total_models
    return float(roc_auc_score(train_frame[target].to_numpy(), oof)), oof, test_pred


## 4. Seed Ensemble Training

In [ ]:
score, oof, pred = _playground_advanced_xgboost_result(train, test, orig, folds=5)
summary = {
    'oof_auc': round(float(score), 5),
    'prediction_rows': int(len(pred)),
    'prediction_min': float(pred.min()),
    'prediction_max': float(pred.max()),
    'prediction_mean': float(pred.mean()),
}
print(json.dumps(summary, indent=2))

## 5. Submission

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'Churn': pred})
submission.to_csv('submission.csv', index=False)
print('submission.csv written to the working directory.')
submission.head()